In [6]:
import os
import sys
import torch
import torchaudio
import re
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display
from tqdm.notebook import tqdm


HOME_PATH = os.path.expanduser("~")   # parent directory of the musicfm folder
sys.path.append(HOME_PATH)

from musicfm.model.musicfm_25hz import MusicFM25Hz

device = "cuda" if torch.cuda.is_available() else "cpu"

#from model.musicfm_25hz import MusicFM25Hz

In [118]:
AUDIO_PATH = "/Users/xinranchen/Downloads/24kbaseline/audio_1000_balanced_original_24k"
OUTPUT_DIR = Path("/Users/xinranchen/Downloads/MusicFM_embeddings/0")

In [119]:
def getID(filename) -> str:
    stem = Path(filename).stem
    m = re.search(r"DB_(?:training|test|validation)-(\d+)_(.+)$", stem)
    if not m:
        raise ValueError(f"Could not extract ID from filename: {filename}")
    return m.group(2), m.group(1)


In [120]:
print(getID("Medley-solos-DB_training-4_05bc0cdb-f7a0-5cbf-fe86-796ca987e411.wav"))

('05bc0cdb-f7a0-5cbf-fe86-796ca987e411', '4')


In [121]:
all_embeddings = {}
for file in tqdm(Path(AUDIO_PATH).iterdir(), desc = "embeddings"):
    # skip hidden files like .DS_Store
    if file.name.startswith("."):
        continue
        
    #print(file.stem)
    file_id, instrument_id = getID(file.stem)
    
    wav, sr = torchaudio.load(str(file))
    wav = wav.to(device)
    musicfm.eval()
    #display(Audio(wav, rate = sr))
    with torch.no_grad():
        emb = musicfm.get_latent(wav, layer_ix=7)
        
    clip_emb = emb.mean(dim=1).squeeze(0).detach().cpu()

    record = {
        "file_id": file_id,
        "source_file": file.name,
        "sample_rate": sr,
        "embedding": clip_emb,
        "instrument_id": instrument_id,
    }

    # save per-file
    torch.save(record, OUTPUT_DIR / f"{file_id}.pt")

    # also collect into one big dictionary
    all_embeddings[file_id] = record

# save one combined file
torch.save(all_embeddings, OUTPUT_DIR / "all_musicfm_embeddings.pt")
print("Saved combined file:", OUTPUT_DIR / "all_musicfm_embeddings.pt")


embeddings: 0it [00:00, ?it/s]

Saved combined file: /Users/xinranchen/Downloads/MusicFM_embeddings/0/all_musicfm_embeddings.pt


In [122]:
saved_embeddings = torch.load(OUTPUT_DIR / "all_musicfm_embeddings.pt")
print(len(saved_embeddings))

1000


In [123]:
item = saved_embeddings["2abb6662-02e8-51b0-ffbd-93880a4db5a9"]
print(item["source_file"])
print(item["file_id"])
print(item["instrument_id"])
print(item["embedding"])

Medley-solos-DB_test-0_2abb6662-02e8-51b0-ffbd-93880a4db5a9.wav
2abb6662-02e8-51b0-ffbd-93880a4db5a9
0
tensor([ 0.9985,  0.2656,  0.2804,  ..., -0.0459, -0.3849,  1.6001])


In [124]:
count = 0
for item in saved_embeddings.values():
    if int(item["instrument_id"]) == 5:
        count+=1
print(count)

125


In [125]:
test, sr = torchaudio.load("/Users/xinranchen/Downloads/24kbaseline/audio_1000_balanced_original_24k/Medley-solos-DB_test-0_8ac94357-d933-5834-f5cd-b610d11c5334.wav")
display(Audio(test, rate = sr))
sr

24000

In [110]:
wav = (torch.rand(4, 24000 * 30) - 0.5) * 2

# load MusicFM
musicfm = MusicFM25Hz(
    is_flash=False,
    stat_path=os.path.join(HOME_PATH,"musicfm","data", "msd_stats.json"),
    model_path=os.path.join(HOME_PATH,"musicfm", "data", "pretrained_msd.pt"),
).to(device)

# to GPUs
# wav = wav.cuda()
# musicfm = musicfm.cuda()

wav = wav.to(device)

# get embeddings
musicfm.eval()
#emb = musicfm.get_latent(wav, layer_ix=7)
with torch.no_grad():
    emb = musicfm.get_latent(wav, layer_ix=7)

In [111]:
emb.shape

torch.Size([4, 750, 1024])

In [112]:
seq_emb = emb.mean(dim=1)
seq_emb.shape

torch.Size([4, 1024])